# B04：决策树与验证选模（辅修学生练习）

2 实验学时，T08 后。用原有超限案例的 120 行开发部分作三折比较；原 validation40 行本次不评价。无独立最终测试。

前 65 分钟完成实验，后 20 分钟填写 project/project.ipynb 中的综合任务计划，最后保存备份。教师发放包不包含 teacher 目录；文件保存在非保护盘并另留副本。

## 1. 数据与共同三折（提供）

中位数填补放进 Pipeline，在每折训练部分重新 fit。树不需要本次线性尺度标准化，但不能省略填补与正确划分。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import StratifiedKFold, cross_validate

candidates = []
for parent in (Path.cwd(), *Path.cwd().parents):
    candidates.extend(
        [
            parent / "data/tree-classification.csv",
            parent / "units/decision-tree/data/tree-classification.csv",
        ]
    )
data_path = next((p for p in candidates if p.is_file()), None)
if data_path is None:
    raise FileNotFoundError("保留目录关系，从单元根目录或labs启动Jupyter")
raw = pd.read_csv(data_path)
dev = raw[raw.split == "train"].reset_index(drop=True)
features = ["planned_load_ratio", "ambient_index"]
X = dev[features]
y = dev.over_limit.to_numpy()
folds = list(StratifiedKFold(n_splits=3, shuffle=True, random_state=42).split(X, y))
assert X.shape == (120, 2)
display(dev.head())
display(X.isna().sum().rename("开发集缺失数"))
for ti, vi in folds:
    assert len(ti) == 80 and len(vi) == 40 and set(ti).isdisjoint(vi)

## 2. 固定候选

深度 1、3、5，其他设置保持一致。不扩大搜索。

In [ ]:
depth_candidates = [1, 3, 5]

In [ ]:
assert depth_candidates == [1, 3, 5], "本次固定比较深度 1、3、5"
assert 2 <= len(depth_candidates) <= 3
assert len(set(depth_candidates)) == len(depth_candidates)
assert all(type(d) is int and 1 <= d <= 6 for d in depth_candidates)
depth_candidates = sorted(depth_candidates)

## 3. 树 Pipeline

定义 build_tree(depth)。步骤名为 imputer 和 tree，填补使用训练中位数。树固定 criterion="gini"、min_samples_leaf=2、random_state=42，只改变 max_depth。构造时不要 fit。

In [ ]:
def build_tree(depth):
    # TODO：把参数depth传给max_depth，完成后删除提示。
    raise NotImplementedError("请设置深度")
    return Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            (
                "tree",
                DecisionTreeClassifier(
                    max_depth=..., min_samples_leaf=2, criterion="gini", random_state=42
                ),
            ),
        ]
    )

## 4. 同折比较（汇总已提供）

run_cv(depth)接收完整 Pipeline、原始 X/y 与共同 folds。accuracy 越大越好；test_score 在这里是折内验证分数，不是最终测试。return_train_score 用于诊断，选择依据仍是验证均分。

后续检查由教师提供，会逐候选、逐折核对实际树设置、填补统计以及训练和验证准确率；保留检查代码。操作说明可回看 T08 第 7 节。

In [ ]:
def run_cv(depth):
    # TODO：补齐流程、输入、标签和共同折；完成后删除提示。
    raise NotImplementedError("请完成验证调用")
    return cross_validate(
        ...,
        ...,
        ...,
        cv=...,
        scoring="accuracy",
        return_train_score=True,
        return_estimator=True,
        error_score="raise",
    )

In [ ]:
# 教师提供：核对实际树设置、逐折填补与准确率，保留检查代码。
def check_tree_settings(model, depth):
    assert isinstance(model, Pipeline), "请返回完整 Pipeline"
    assert list(model.named_steps) == ["imputer", "tree"]
    imputer = model.named_steps["imputer"]
    tree = model.named_steps["tree"]
    assert isinstance(imputer, SimpleImputer) and imputer.strategy == "median"
    assert isinstance(tree, DecisionTreeClassifier)
    assert tree.max_depth == depth, "实际最大深度必须等于当前候选，不能写死"
    assert tree.min_samples_leaf == 2, "每个叶至少保留 2 条训练记录"
    assert tree.criterion == "gini" and tree.random_state == 42


def check_tree_run(depth, run):
    assert len(run["estimator"]) == len(folds) == 3
    for key in ("test_score", "train_score"):
        scores = np.asarray(run[key])
        assert scores.shape == (3,) and np.isfinite(scores).all()
    for k, ((ti, vi), model) in enumerate(zip(folds, run["estimator"])):
        check_tree_settings(model, depth)
        medians = X.iloc[ti].median().to_numpy()
        np.testing.assert_allclose(
            model.named_steps["imputer"].statistics_,
            medians,
            err_msg="填补值应来自当前候选、当前折的训练行",
        )
        tree = model.named_steps["tree"]
        assert tree.tree_.n_node_samples[0] == len(ti) == 80
        assert tree.get_depth() <= depth
        leaf_sizes = tree.tree_.n_node_samples[tree.tree_.children_left == -1]
        assert np.all(leaf_sizes >= 2)
        for key, indices in (("train_score", ti), ("test_score", vi)):
            accuracy = np.mean(model.predict(X.iloc[indices]) == y[indices])
            np.testing.assert_allclose(
                run[key][k],
                accuracy,
                rtol=0,
                atol=1e-12,
                err_msg="分数必须是对应训练或验证行的准确率",
            )


runs = {}
rows = []
for depth in depth_candidates:
    prototype = build_tree(depth)
    check_tree_settings(prototype, depth)
    assert not hasattr(prototype.named_steps["imputer"], "statistics_")
    assert not hasattr(prototype.named_steps["tree"], "tree_")
    run = run_cv(depth)
    check_tree_run(depth, run)
    runs[depth] = run
    s = run["test_score"]
    rows.append(
        {
            "depth": run["estimator"][0].named_steps["tree"].max_depth,
            "train_mean": run["train_score"].mean(),
            "fold1": s[0],
            "fold2": s[1],
            "fold3": s[2],
            "validation_mean": s.mean(),
            "score_std": s.std(ddof=0),
        }
    )
results = pd.DataFrame(rows)
display(results)
assert np.isfinite(results.to_numpy()).all()
baseline_scores = []
for ti, vi in folds:
    majority = np.bincount(y[ti], minlength=2).argmax()
    baseline_scores.append(np.mean(y[vi] == majority))
print("同折多数类基线均分：", np.mean(baseline_scores))
fig, ax = plt.subplots(figsize=(6, 3.3))
results.set_index("depth")[["train_mean", "validation_mean"]].plot.bar(ax=ax, rot=0)
ax.set(ylabel="Accuracy", ylim=(0, 1))
fig.tight_layout()
plt.show()

## 5. 选定并审查

选 validation_mean 最大的深度；并列选更浅者。score_std 仅描述折间波动，不是置信区间。审查首折填补统计，不使用全 120 行填补后再分折。

In [ ]:
# TODO：补入用于选择的验证均分列名；并列按深度升序。
raise NotImplementedError("请按验证结果选择")
selected_depth = int(
    results.sort_values([..., "depth"], ascending=[False, True]).iloc[0]["depth"]
)

In [ ]:
expected = int(
    results.sort_values(["validation_mean", "depth"], ascending=[False, True])
    .iloc[0]
    .depth
)
assert selected_depth == expected
ti, vi = folds[0]
fold_model = runs[selected_depth]["estimator"][0]
np.testing.assert_allclose(
    fold_model.named_steps["imputer"].statistics_, X.iloc[ti].median().to_numpy()
)
assert fold_model.named_steps["tree"].tree_.n_node_samples[0] == 80
final_tree = build_tree(selected_depth).fit(X, y)
check_tree_settings(final_tree, selected_depth)
assert final_tree.named_steps["tree"].tree_.n_node_samples[0] == len(X) == 120
np.testing.assert_allclose(
    final_tree.named_steps["imputer"].statistics_, X.median().to_numpy()
)
print("选定深度：", selected_depth, "全开发集重拟合；不计算原validation表现。")
fig, ax = plt.subplots(figsize=(12, 6))
plot_tree(
    final_tree.named_steps["tree"],
    feature_names=features,
    class_names=["normal", "over_limit"],
    max_depth=2,
    filled=True,
    precision=2,
    fontsize=8,
    ax=ax,
)
ax.set_title("Selected tree: first 3 levels only; deeper branches may be truncated")
fig.tight_layout()
plt.show()

## 6. 实验短答

1. 引用训练和验证均分说明深度选择；为何不只选训练最高？
2. 首折填补值来自哪些行？树不用标准化是否意味着可全表填补？
3. 沿图解释一个分支；叶内类别比例是否是已校准真实风险？

**我的回答：**（填写）

## 7. 综合任务启动（20 分钟，不训练新模型）

打开同包 project/project.ipynb 与 project/README.md，登记质量不合格预测的问题、输入、基线、指标及划分计划。它是新开发数据，不是本实验旧验证集。

本次只写草稿，后续在同一 Notebook 继续，不另交重复报告。实验文件和项目文件保存非保护盘并另留副本。教师保管的测试资料不发学生、不用于选模。

本次项目计划先列一个候选，并说明预测时点、合法特征、一个基线、主要指标及划分理由。